# 2.2 State (adaptado a lab local)

**Qué se ve aquí:** el *state* del agente se puede extender (`state_schema`) y las tools pueden **escribirlo** (`Command(update=...)`) y **leerlo** (`runtime.state`). A diferencia del runtime context, el state **cambia** durante la ejecución y se guarda en el checkpoint del thread.

**Convención de este fork:** 🔸 ORIGINAL DEL CURSO (comentada, con el motivo) → 🟢 ADAPTADO LOCAL (la que corre con Ollama). Celdas sin marca = iguales al curso.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (raíz del repo) y definir el modelo local.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [1]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-2/ y local_model.py en la raíz del repo.
#         MODEL reemplaza a "gpt-5-nano" en todas las celdas (gemma4: soporta tools, el más rápido del lab).
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model, OLLAMA_URL

MODEL = get_model("gemma4:latest")

In [2]:
# Sin cambios
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [3]:
# Sin cambios
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent
# from langgraph.checkpoint.memory import InMemorySaver
#
# agent = create_agent(
#     "gpt-5-nano",
#     tools=[update_favourite_colour],
#     checkpointer=InMemorySaver(),
#     state_schema=CustomState
# )

In [4]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" → MODEL (gemma4 en Ollama, definido en la celda de setup).
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=MODEL,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [6]:
# Sin cambios
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [7]:
# Sin cambios
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='4d1ff8d1-9ba2-4d91-894a-2d667df42c71'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:latest', 'created_at': '2026-10-05T00:28:02.6101225Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2549053900, 'load_duration': 22107100, 'prompt_eval_count': 82, 'prompt_eval_duration': 1624179000, 'eval_count': 21, 'eval_duration': 894504000, 'logprobs': None, 'model_name': 'gemma4:latest', 'model_provider': 'ollama'}, id='lc_run--01a10975-d635-7981-a594-160a42dfa7df-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'ce4b06c5-331f-4d74-865b-642f831205f3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 82, 'output_tokens': 21, 'total_tokens': 103}),
              ToolMessage(content='Successfully updated favourite colour', 

In [8]:
# Sin cambios
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='2773d630-d6c8-4967-8488-7bd066240e57'),
              AIMessage(content="I'm doing well, thank you for asking! How can I help you today?", additional_kwargs={}, response_metadata={'model': 'gemma4:latest', 'created_at': '2026-10-05T00:29:26.7691683Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2560948000, 'load_duration': 4825700, 'prompt_eval_count': 83, 'prompt_eval_duration': 1653185000, 'eval_count': 19, 'eval_duration': 900046000, 'logprobs': None, 'model_name': 'gemma4:latest', 'model_provider': 'ollama'}, id='lc_run--01a10977-1ef5-7913-9897-3037d3a64fc2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 83, 'output_tokens': 19, 'total_tokens': 102})]}


## Read state

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# @tool
# def read_favourite_colour(runtime: ToolRuntime) -> str:
#     """Read the favourite colour of the user from the state."""
#     try:
#         return runtime.state["favourite_colour"]
#     except KeyError:
#         return "No favourite colour found in state"
#
# agent = create_agent(
#     "gpt-5-nano",
#     tools=[update_favourite_colour, read_favourite_colour],
#     checkpointer=InMemorySaver(),
#     state_schema=CustomState
# )

In [9]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" → MODEL (gemma4 en Ollama, definido en la celda de setup).
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    model=MODEL,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [10]:
# Sin cambios
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='4a7b1f3a-8803-490d-b724-d67352ab0257'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:latest', 'created_at': '2026-10-05T00:31:21.1039195Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3151302100, 'load_duration': 6361100, 'prompt_eval_count': 117, 'prompt_eval_duration': 2237662000, 'eval_count': 21, 'eval_duration': 902788000, 'logprobs': None, 'model_name': 'gemma4:latest', 'model_provider': 'ollama'}, id='lc_run--01a10978-db4c-7721-9064-0ed996a7fc10-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'ca7457f1-b120-4598-8de1-2c3a356db218', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_tokens': 21, 'total_tokens': 138}),
              ToolMessage(content='Successfully updated favourite colour',

In [11]:
# Sin cambios
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='4a7b1f3a-8803-490d-b724-d67352ab0257'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:latest', 'created_at': '2026-10-05T00:31:21.1039195Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3151302100, 'load_duration': 6361100, 'prompt_eval_count': 117, 'prompt_eval_duration': 2237662000, 'eval_count': 21, 'eval_duration': 902788000, 'logprobs': None, 'model_name': 'gemma4:latest', 'model_provider': 'ollama'}, id='lc_run--01a10978-db4c-7721-9064-0ed996a7fc10-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'ca7457f1-b120-4598-8de1-2c3a356db218', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_tokens': 21, 'total_tokens': 138}),
              ToolMessage(content='Successfully updated favourite colour',